# 01 · Análisis Exploratorio de Datos (EDA)
**Universidad Alfonso X el Sabio — Bases de Datos e Inteligencia Artificial 2025-2026**

Este notebook explora el dataset completo de 50.001 pacientes antes de cualquier decisión de modelado.  
Su propósito es descriptivo: entender distribuciones, detectar anomalías, cuantificar el desbalance  
y documentar qué variables tienen poder predictivo real y cuáles deben descartarse.

Las conclusiones de este notebook alimentan directamente las decisiones de `02_feature_engineering.ipynb`.

## 00 · Imports y configuración

In [ ]:
import sys, os
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid', palette='muted')

DATA_DIR = r'../Base de datos-20260512'
FIG_DIR  = r'../outputs/figures'
os.makedirs(FIG_DIR, exist_ok=True)

TARGET = 'cancer'
print('Entorno listo.')

## 01 · Carga de todos los ficheros

In [ ]:
# En el EDA cargamos los 6 ficheros incluyendo ECONOMICOS,
# para poder demostrar empíricamente por qué sus variables son leakage.
bioquim   = pd.read_csv(f'{DATA_DIR}/CASOCANCER_01_BIOQUIMICOS.csv')
clinicos  = pd.read_csv(f'{DATA_DIR}/CASOCANCER_02_CLINICOS.csv')
geneticos = pd.read_csv(f'{DATA_DIR}/CASOCANCER_03_GENETICOS.csv')
economicos= pd.read_csv(f'{DATA_DIR}/CASOCANCER_04_ECONOMICOS.csv')
generales = pd.read_csv(f'{DATA_DIR}/CASOCANCER_05_GENERALES.csv')
sociodemo = pd.read_csv(f'{DATA_DIR}/CASOCANCER_06_SOCIODEMOGRAFICOS.csv')

frames = {
    '01_BIOQUIMICOS' : bioquim,
    '02_CLINICOS'    : clinicos,
    '03_GENETICOS'   : geneticos,
    '04_ECONOMICOS'  : economicos,
    '05_GENERALES'   : generales,
    '06_SOCIODEMO'   : sociodemo,
}

print(f'{'Fichero':<22} {'Filas':>7}  {'Cols':>4}  {'Nulos':>6}')
print('-' * 45)
for name, df_ in frames.items():
    print(f'{name:<22} {df_.shape[0]:>7,}  {df_.shape[1]:>4}  {df_.isnull().sum().sum():>6}')

In [ ]:
# Merge completo para análisis conjunto
df = (
    bioquim
    .merge(clinicos,   on='paciente_id')
    .merge(geneticos,  on='paciente_id')
    .merge(economicos, on='paciente_id')
    .merge(generales,  on='paciente_id')
    .merge(sociodemo,  on='paciente_id')
)

print(f'Dataset completo: {df.shape[0]:,} pacientes × {df.shape[1]} variables')
print(f'Nulos totales: {df.isnull().sum().sum()}')
df.dtypes.to_frame('dtype').T

## 02 · Variable objetivo — `cancer`

In [ ]:
# Cuantificamos el desbalance de clases antes de cualquier otra cosa.
# El ratio ~4.2:1 es moderado: suficiente para necesitar corrección
# (class_weight / pos_weight), pero no tan extremo como para requerir SMOTE.
vc = df[TARGET].value_counts().sort_index()
ratio = vc[0] / vc[1]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Barras
bars = axes[0].bar(
    ['No cáncer (0)', 'Cáncer (1)'], vc.values,
    color=['steelblue', 'tomato'], edgecolor='white', linewidth=1.2
)
axes[0].set_title('Distribución del target', fontsize=12)
axes[0].set_ylabel('Pacientes')
for bar, v in zip(bars, vc.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, v + 400,
        f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=10, fontweight='bold'
    )

# Pie
axes[1].pie(
    vc.values, labels=['No cáncer', 'Cáncer'],
    colors=['steelblue', 'tomato'], autopct='%1.1f%%',
    startangle=90, wedgeprops=dict(edgecolor='white', linewidth=1.5)
)
axes[1].set_title('Proporción del target', fontsize=12)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_01_target.png', bbox_inches='tight')
plt.show()

print(f'Clase 0 (No cáncer): {vc[0]:,}  ({vc[0]/len(df):.2%})')
print(f'Clase 1 (Cáncer)   : {vc[1]:,}  ({vc[1]/len(df):.2%})')
print(f'Ratio desbalance   : {ratio:.2f}:1')

## 03 · Variables bioquímicas

In [ ]:
# Estadísticas descriptivas por clase: buscamos diferencias de media/mediana
# que indiquen poder predictivo. Glucosa, hemoglobina y leucocitos tienen
# pesos explícitos en el modelo generativo (+1.2, +0.9, +0.7 respectivamente).
bio_cols = ['glucosa', 'colesterol', 'trigliceridos', 'hemoglobina',
            'leucocitos', 'plaquetas', 'creatinina']

desc = df.groupby(TARGET)[bio_cols].mean().T.round(2)
desc.columns = ['media cancer=0', 'media cancer=1']
desc['diferencia'] = (desc['media cancer=1'] - desc['media cancer=0']).round(2)
desc['diferencia %'] = ((desc['diferencia'] / desc['media cancer=0']) * 100).round(1)
print(desc.to_string())

In [ ]:
# Distribuciones superpuestas: un desplazamiento visible entre cancer=0 y cancer=1
# confirma poder discriminativo. Curvas muy solapadas → feature de bajo valor.
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for i, col in enumerate(bio_cols):
    for val, color, label in [(0, 'steelblue', 'No cáncer'), (1, 'tomato', 'Cáncer')]:
        axes[i].hist(
            df[df[TARGET] == val][col], bins=45,
            alpha=0.55, color=color, label=label, density=True
        )
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)

axes[-1].set_visible(False)   # celda vacía
plt.suptitle('Variables bioquímicas — distribución por diagnóstico', y=1.01, fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_02_bioquimicas.png', bbox_inches='tight')
plt.show()

In [ ]:
# Boxplots: permiten ver mediana, IQR y outliers en una sola gráfica
fig, axes = plt.subplots(1, len(bio_cols), figsize=(18, 5))

for ax, col in zip(axes, bio_cols):
    data_plot = [df[df[TARGET]==0][col], df[df[TARGET]==1][col]]
    bp = ax.boxplot(data_plot, patch_artist=True, widths=0.5,
                    medianprops=dict(color='black', linewidth=2))
    bp['boxes'][0].set_facecolor('steelblue')
    bp['boxes'][1].set_facecolor('tomato')
    ax.set_xticklabels(['No\ncáncer', 'Cáncer'], fontsize=9)
    ax.set_title(col, fontsize=10)

plt.suptitle('Boxplots bioquímica por diagnóstico', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_03_bioquimicas_boxplot.png', bbox_inches='tight')
plt.show()

## 04 · Variables genéticas (mutaciones oncogénicas)

In [ ]:
# Las mutaciones son las features con mayor poder predictivo del dataset.
# El lift (diferencia de prevalencia entre cancer=1 y cancer=0) indica
# cuánto aumenta la probabilidad de cáncer al ser portador.
# mut_ALK se incluye aquí solo para mostrar empíricamente que su lift es ~0.
mut_cols = ['mut_BRCA1', 'mut_TP53', 'mut_EGFR', 'mut_KRAS',
            'mut_PIK3CA', 'mut_ALK', 'mut_BRAF']

prev = pd.DataFrame({
    'Prevalencia cancer=0 (%)': df[df[TARGET]==0][mut_cols].mean() * 100,
    'Prevalencia cancer=1 (%)': df[df[TARGET]==1][mut_cols].mean() * 100,
}).round(2)
prev['Lift (pp)'] = (prev['Prevalencia cancer=1 (%)'] - prev['Prevalencia cancer=0 (%)']).round(2)
prev['Lift relativo (x)'] = (prev['Prevalencia cancer=1 (%)'] / prev['Prevalencia cancer=0 (%)']).round(2)
print(prev.sort_values('Lift (pp)', ascending=False).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Prevalencia por clase
prev_plot = prev[['Prevalencia cancer=0 (%)', 'Prevalencia cancer=1 (%)']]
prev_plot.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'],
               edgecolor='white', rot=35)
axes[0].set_title('Prevalencia de mutaciones por diagnóstico (%)')
axes[0].set_ylabel('Prevalencia (%)')
axes[0].legend()

# Lift (pp)
lift_sorted = prev['Lift (pp)'].sort_values(ascending=True)
colors = ['lightcoral' if v < 1 else 'tomato' for v in lift_sorted]
lift_sorted.plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Lift de mutaciones sobre cancer=1 (pp)')
axes[1].set_xlabel('Diferencia de prevalencia (puntos porcentuales)')
# Señalar mut_ALK
axes[1].annotate('← sin lift\n  (excluir)', xy=(lift_sorted['mut_ALK'], 5),
                 fontsize=8, color='gray')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_04_mutaciones.png', bbox_inches='tight')
plt.show()

## 05 · Variables clínicas (comorbilidades)

In [ ]:
# Las comorbilidades correlacionan con cancer por diseño del modelo generativo.
# El riesgo de leakage es INDIRECTO (son factores de riesgo, no consecuencias),
# por lo que se pueden incluir si el lift justifica su aportación.
# enfermedad_cardiaca, asma y epoc tienen lift < 2 pp → candidatas a descartar.
clin_cols = ['diabetes', 'hipertension', 'obesidad',
             'enfermedad_cardiaca', 'asma', 'epoc']

clin_prev = pd.DataFrame({
    'Prevalencia cancer=0 (%)': df[df[TARGET]==0][clin_cols].mean() * 100,
    'Prevalencia cancer=1 (%)': df[df[TARGET]==1][clin_cols].mean() * 100,
}).round(1)
clin_prev['Lift (pp)'] = (clin_prev['Prevalencia cancer=1 (%)'] - clin_prev['Prevalencia cancer=0 (%)']).round(1)
print(clin_prev.sort_values('Lift (pp)', ascending=False).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

x = range(len(clin_cols))
w = 0.35
bars0 = ax.bar([i - w/2 for i in x],
               [df[df[TARGET]==0][c].mean()*100 for c in clin_cols],
               width=w, label='No cáncer', color='steelblue', edgecolor='white')
bars1 = ax.bar([i + w/2 for i in x],
               [df[df[TARGET]==1][c].mean()*100 for c in clin_cols],
               width=w, label='Cáncer', color='tomato', edgecolor='white')

ax.set_xticks(list(x))
ax.set_xticklabels(clin_cols, rotation=20, ha='right')
ax.set_ylabel('Prevalencia (%)')
ax.set_title('Comorbilidades — prevalencia por diagnóstico')
ax.legend()

# Anotar lift
for i, col in enumerate(clin_cols):
    lift_val = clin_prev.loc[col, 'Lift (pp)']
    color = 'darkgreen' if lift_val >= 5 else 'gray'
    ax.text(i, max(df[df[TARGET]==0][col].mean()*100,
                   df[df[TARGET]==1][col].mean()*100) + 1,
            f'+{lift_val} pp', ha='center', fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_05_comorbilidades.png', bbox_inches='tight')
plt.show()

## 06 · Variables de hábitos de vida

In [ ]:
# alcohol es constante en todo el dataset → varianza cero → se descarta.
# vive es consecuencia del diagnóstico → leakage → se descarta.
print('== alcohol (debe ser constante) ==')
print(df['alcohol'].value_counts())
print(f'\nVarianza: {df["alcohol"].var():.4f}  → constante, sin información')

print('\n== vive (supervivencia al cierre del seguimiento) ==')
print(df.groupby(TARGET)['vive'].mean().rename('tasa supervivencia').to_frame())
print('→ vive depende de cancer, no al revés → leakage si se usa como feature')

In [ ]:
# fumador: factor de riesgo causal con peso +1.5 en el modelo generativo
# actividad_fisica: factor protector con pesos -1.2 (Alta) y -0.6 (Moderada)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# fumador
fum_prev = df.groupby(TARGET)['fumador'].mean() * 100
axes[0].bar(['No cáncer', 'Cáncer'], fum_prev.values,
            color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_ylabel('% fumadores')
axes[0].set_title('Tabaquismo por diagnóstico')
for i, v in enumerate(fum_prev.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

# actividad_fisica
act_ct = df.groupby([TARGET, 'actividad_fisica']).size().unstack(fill_value=0)
act_pct = act_ct.div(act_ct.sum(axis=1), axis=0) * 100
act_pct = act_pct[['Baja', 'Moderada', 'Alta']]  # orden causal
act_pct.T.plot(kind='bar', ax=axes[1], color=['steelblue', 'tomato'],
               edgecolor='white', rot=0)
axes[1].set_ylabel('% pacientes')
axes[1].set_title('Actividad física por diagnóstico')
axes[1].legend(['No cáncer', 'Cáncer'])

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_06_habitos.png', bbox_inches='tight')
plt.show()

## 07 · Variables económicas — demostración de data leakage

Estas variables **no deben usarse como features**. Son consecuencia del diagnóstico, no causas.  
Si las incluyéramos, el modelo aprendería patrones triviales (*"más días en hospital → cáncer"*)  
que son inútiles en predicción real donde aún no se ha diagnosticado al paciente.

In [ ]:
# Convertimos coste_total y coste_farmaco a float (usan coma decimal en el CSV)
for col in ['coste_total', 'coste_farmaco']:
    df[col] = df[col].astype(str).str.replace(',', '.').astype(float)

eco_cols = ['coste_total', 'coste_farmaco', 'num_ingresos', 'dias_hospital']

eco_desc = df.groupby(TARGET)[eco_cols].mean().T.round(1)
eco_desc.columns = ['media cancer=0', 'media cancer=1']
eco_desc['diferencia %'] = ((eco_desc['media cancer=1'] - eco_desc['media cancer=0'])
                              / eco_desc['media cancer=0'] * 100).round(1)
print('Medias por diagnóstico — variables económicas:')
print(eco_desc.to_string())
print('\n→ Diferencias enormes: son consecuencias del diagnóstico, no causas.')
print('  Incluirlas sería AUC-ROC artificialmente perfecto → leakage severo.')

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

for ax, col in zip(axes, eco_cols):
    data_0 = df[df[TARGET]==0][col]
    data_1 = df[df[TARGET]==1][col]
    ax.hist(data_0, bins=40, alpha=0.55, color='steelblue', density=True, label='No cáncer')
    ax.hist(data_1, bins=40, alpha=0.55, color='tomato',    density=True, label='Cáncer')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Variables económicas — distribución por diagnóstico\n'
             '(diferencias claras = leakage, NO usar como features)', y=1.03, fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_07_economicas_leakage.png', bbox_inches='tight')
plt.show()

## 08 · Variables sociodemográficas

In [ ]:
# edad es la única sociodemográfica presente en el modelo generativo (peso +0.4 para edad>55).
# El resto (nivel_educativo, zona, estado_civil...) no aparece en el generativo.
# Las analizamos para verificar si muestran alguna señal antes de descartarlas.

# edad — distribución y media por clase
print('== edad ==')
print(df.groupby(TARGET)['edad'].describe().round(1).T.to_string())

# Categóricas: prevalencia de cancer=1 por categoría
cat_cols = ['nivel_educativo', 'nivel_ingresos', 'zona', 'estado_civil']
print()
for col in cat_cols:
    rate = df.groupby(col)[TARGET].mean().sort_values(ascending=False) * 100
    print(f'-- {col} (prevalencia cáncer %) --')
    print(rate.round(2).to_string())
    print()

In [ ]:
# edad: distribución por clase
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for val, color, label in [(0, 'steelblue', 'No cáncer'), (1, 'tomato', 'Cáncer')]:
    axes[0].hist(df[df[TARGET]==val]['edad'], bins=35, alpha=0.55,
                 color=color, label=label, density=True)
axes[0].set_title('Distribución de edad por diagnóstico')
axes[0].set_xlabel('Edad'); axes[0].legend()

# Prevalencia de cáncer por franja de edad (binning de 10 años)
df['edad_grupo'] = pd.cut(df['edad'], bins=range(20, 95, 10))
age_rate = df.groupby('edad_grupo', observed=True)[TARGET].mean() * 100
age_rate.plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='white', rot=30)
axes[1].set_title('Prevalencia de cáncer por franja de edad')
axes[1].set_ylabel('% cáncer')
axes[1].set_xlabel('Grupo de edad')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_08_edad.png', bbox_inches='tight')
plt.show()
df.drop(columns='edad_grupo', inplace=True)

## 09 · Mapa de correlaciones

In [ ]:
# Correlación de Pearson entre features numéricas/binarias y el target.
# Las mutaciones y fumador lideran; las bioquímicas tienen señal más débil.
# Las variables económicas quedan excluidas de aquí por ser leakage.
corr_cols = [
    'glucosa', 'colesterol', 'trigliceridos', 'hemoglobina', 'leucocitos',
    'plaquetas', 'creatinina', 'edad',
    'mut_BRCA1', 'mut_TP53', 'mut_EGFR', 'mut_KRAS', 'mut_PIK3CA', 'mut_ALK', 'mut_BRAF',
    'fumador', 'diabetes', 'hipertension', 'obesidad',
    'enfermedad_cardiaca', 'asma', 'epoc',
    TARGET
]

# Codificar actividad_fisica para incluirla
df['act_fisica_num'] = df['actividad_fisica'].map({'Baja': 0, 'Moderada': 1, 'Alta': 2})
corr_cols_ext = corr_cols[:-1] + ['act_fisica_num', TARGET]

corr_matrix = df[corr_cols_ext].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.3, ax=ax, annot_kws={'size': 7}
)
ax.set_title('Mapa de correlaciones (Pearson)', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_09_correlaciones.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlaciones con el target ordenadas de mayor a menor
target_corr = corr_matrix[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 8))
colors = ['tomato' if v > 0 else 'steelblue' for v in target_corr]
target_corr.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Correlación con `{TARGET}` (Pearson)', fontsize=12)
ax.set_xlabel('Correlación')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/eda_10_corr_target.png', bbox_inches='tight')
plt.show()

df.drop(columns='act_fisica_num', inplace=True)

## 10 · Resumen y conclusiones del EDA

In [ ]:
print('=' * 65)
print('RESUMEN EDA — DECISIONES PARA FEATURE ENGINEERING')
print('=' * 65)

decisiones = [
    ('INCLUIR',   'glucosa, hemoglobina, leucocitos',    'Lift bioquímico + peso en modelo generativo'),
    ('INCLUIR',   'colesterol, trigliceridos, plaquetas, creatinina', 'Predictores bioquímicos causales'),
    ('INCLUIR',   'mut_BRCA1, mut_TP53, mut_KRAS',       'Lift genético > 10 pp'),
    ('INCLUIR',   'mut_EGFR, mut_PIK3CA, mut_BRAF',      'Lift genético 3-7 pp'),
    ('INCLUIR',   'fumador',                              'Lift +19 pp, peso +1.5 en generativo'),
    ('INCLUIR',   'actividad_fisica',                     'Factor protector causal (ordinal: Baja<Mod<Alta)'),
    ('INCLUIR',   'edad',                                 'Único sociodemográfico en el modelo generativo'),
    ('INCLUIR',   'obesidad, hipertension, diabetes',     'Lift > 7 pp, presentes en modelo generativo'),
    ('EXCLUIR',   'mut_ALK',                              'Lift ~0 pp (4.92 % vs 4.89 %)'),
    ('EXCLUIR',   'enfermedad_cardiaca, asma, epoc',      'Lift < 2 pp, fuera del modelo generativo'),
    ('EXCLUIR',   'alcohol',                              'Constante (100 % = 1), varianza cero'),
    ('EXCLUIR',   'vive',                                 'Consecuencia del diagnóstico → leakage'),
    ('EXCLUIR',   'coste_total, coste_farmaco, num_ingresos, dias_hospital', 'Consecuencias → leakage severo'),
    ('EXCLUIR',   'nivel_educativo, zona, estado_civil, nivel_ingresos, num_hijos, distancia_hospital_km',
                  'Ausentes en generativo, sin señal observada'),
]

for decision, variables, motivo in decisiones:
    tag = '✅' if decision == 'INCLUIR' else '❌'
    print(f'\n{tag} {decision}: {variables}')
    print(f'   Motivo: {motivo}')

print('\n' + '=' * 65)
print(f'Total features finales: 19')
print('Continúa en → 02_feature_engineering.ipynb')